# 01 - From a timetable to the frozen Transperth network

**Opening demonstration (about two minutes).** Start with the map, hide the bus
layer, then find a central interchange and a branch station. The question is
whether a connected railway remains useful after its important facilities close.

This notebook uses committed inputs, not today's timetable. The snapshot is
**Monday 5 October 2026, [07:00, 09:00), Australia/Perth**. It contains 271
selected rail trips and 4,324 selected bus trips. Trip counts describe scheduled
services, not passengers. The code checks the processed-file hashes first;
rebuilding the raw feed is a separate operation described in [docs/data.md](../docs/data.md).

In [1]:
from pathlib import Path
import sys
import time

started = time.perf_counter()
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/transperth/config.py').is_file())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, display
from transperth.network import load_rail_graph, load_bus_coverage
from transperth.multilayer import build_layers, add_backup_edges, add_walking_transfers
from transperth.notebook_support import verify_processed, network_map_html
from transperth.plotting import apply_style

manifest = verify_processed(ROOT)
rail = load_rail_graph()
coverage = load_bus_coverage()
backups = pd.read_csv(ROOT / 'data/processed/backup_edges.csv',
                      dtype={'station_a': str, 'station_b': str})
apply_style()
assert (rail.number_of_nodes(), rail.number_of_edges()) == (86, 85)
assert nx.is_tree(rail)
display(pd.DataFrame([{
    'service_date': manifest['service_date'], 'window': '[07:00, 09:00) Perth',
    'mapped_stations': len(rail), 'rail_segments': rail.number_of_edges(),
    'rail_trips': manifest['rail_trips'], 'bus_trips': manifest['bus_trips'],
    'verified_bus_pairs': len(backups), 'rail_graph_is_tree': nx.is_tree(rail),
}]))

,service_date,window,mapped_stations,rail_segments,rail_trips,bus_trips,verified_bus_pairs,rail_graph_is_tree
0,2026-10-05,"[07:00, 09:00) Perth",86,85,271,4324,4,True


## What becomes a node or an edge?

The preparation pipeline selects active services from `calendar.txt` and
`calendar_dates.txt`, filters `stop_times.txt` to the morning window, maps rail
platforms to station IDs, and counts traversals of **adjacent mapped stations**.
Express trips are expanded along the mapped corridor: a skipped stop does not
create a shortcut. The audit removes eight such shortcuts and three off-map
connections from the old 96-edge prototype. The retained graph has 86 vertices
and 85 undirected edges. Its connectivity and acyclicity are checked above.

For a rail segment, distance uses station coordinates and the great-circle law:

$$
d_{ij}=2R\arcsin\sqrt{\sin^2\!\left(\frac{\phi_j-\phi_i}{2}\right)+\cos\phi_i\cos\phi_j\sin^2\!\left(\frac{\lambda_j-\lambda_i}{2}\right)},\quad R=6{,}371{,}000\ \mathrm{m}.
$$

Angles are radians. Frequency is a scheduled traversal rate, not a headway for
every individual route or direction:

$$
f_{ij}=\frac{\text{selected traversals of segment }\{i,j\}}{120\ \mathrm{min}}.
$$

`trips_served` counts visits in each selected trip's full stop sequence;
`am_peak_stops` counts station departures actually inside the window. They have
different denominators. A service can extend beyond 09:00. Showgrounds remains
on the mapped corridor with zero weekday stop events. We do not erase it simply
because the window has no scheduled stop there.

Source and regeneration: [scripts/run_data.py](../scripts/run_data.py),
[topology_audit.csv](../data/processed/topology_audit.csv), and
[the column dictionary](../docs/data.md).

In [2]:
stations = pd.DataFrame([{'station_id': node, **attrs}
                         for node, attrs in rail.nodes(data=True)])
stations['lines'] = stations['lines'].map('; '.join)
columns = ['station_id', 'name', 'degree', 'lines', 'trips_served', 'am_peak_stops']
with pd.option_context('display.max_rows', 100):
    display(stations[columns].sort_values(['name', 'station_id']).reset_index(drop=True))
display(coverage[['station_id', 'name', 'n_bus_stops', 'n_bus_routes']].head(10))

,station_id,name,degree,lines,trips_served,am_peak_stops
0,148,Airport Central Stn,2,Airport Line,26,20
1,153,Alkimos Stn,2,Yanchep Line,41,26
2,43,Armadale Stn,2,Armadale Line,26,19
3,7,Ashfield Stn,2,Midland Line,24,20
4,138,Aubin Grove Stn,2,Mandurah Line,39,26
5,158,Ballajura Stn,2,Ellenbrook Line,24,20
6,39,Bassendean Stn,2,Midland Line,24,20
7,23,Bayswater Stn,4,Airport Line; Ellenbrook Line; Midland Line,74,60
8,28,Beckenham Stn,3,Armadale Line; Thornlie-Cockburn Line,48,36
9,82,Bull Creek Stn,2,Mandurah Line,61,47


,station_id,name,n_bus_stops,n_bus_routes
0,1,North Fremantle Stn,8,6
1,100,Warwick Stn,10,18
2,101,Wellard Stn,8,3
3,102,Whitfords Stn,9,19
4,12,Queens Park Stn,6,3
5,133,Butler Stn,10,8
6,135,Elizabeth Quay Stn,43,41
7,138,Aubin Grove Stn,6,6
8,140,Perth Stadium Stn,0,0
9,147,Redcliffe Stn,9,7


## Two facilities at one place

A persistent terminal `T:i` and a closable rail facility `R:i` represent each
station. Rail edges join facilities; substitute buses join terminals. Thus a
rail failure need not erase the bus pickup point. Access takes one assumed
minute. We explicitly add the five-minute Perth / Perth Underground walking
interchange; it is not a rail edge. This baseline activates the **four verified
bus pairs**. Coverage within 400 m is evidence of local bus availability; it does
not by itself establish a direct station-to-station bus edge. The larger
GTFS candidate pool is studied in Notebook 04.

The rail time approximation uses 40 km/h, not measured run times:

$$
t^{\mathrm{rail}}_{ij}=\max\left(0.1,\frac{d_{ij}/1000}{40}\,60\right)\ \mathrm{min}.
$$

Bus times come from the frozen manual evidence and include waiting and walking.
They are symmetric effective times in this undirected scenario, although the
underlying recorded directions differ. See [docs/model.md, section 5](../docs/model.md).

In [3]:
layered = add_walking_transfers(add_backup_edges(build_layers(rail), backups), [('56', '64')])
layer_counts = pd.Series([d['mode'] for _, _, d in layered.edges(data=True)]).value_counts()
display(pd.DataFrame({'edge_mode': layer_counts.index, 'count': layer_counts.values}))
assert layered.number_of_nodes() == 2 * len(rail)
assert layer_counts['rail'] == 85 and layer_counts['access'] == 86
assert layer_counts['backup'] == 4 and layer_counts['walk'] == 1
display(backups[['station_a', 'station_b', 'minutes', 'kind', 'route_or_road']])
display(HTML(network_map_html(rail, backups=backups,
                            title='86 stations: rail corridors and four verified bus links')))

,edge_mode,count
0,access,86
1,rail,85
2,backup,4
3,walk,1


,station_a,station_b,minutes,kind,route_or_road
0,56,88,38,existing_bus,56->88: 990; 88->56: 990
1,56,23,38,existing_bus,56->23: 40; 23->56: 41
2,56,48,38,existing_bus,56->48: 960; 48->56: 960
3,73,83,42,existing_bus,73->83: 998 + 101; 83->73: 101 + 999


C:\Users\TAKO\Documents\Codex\2026-10-04\concept-2-transperth-network-failure-v-2\work\qa_deps\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


The map uses station coordinates in a local projection. It shows endpoint links,
not streets or vehicle paths. Pan, zoom, toggle layers, and hover a station.
The two rail facilities at Perth are distinct even though passengers can walk
between their terminals. Trust this notebook and run it in JupyterLab for the
iframe controls; all map assets are embedded and work offline.

In [4]:
line_names = sorted({line for _, attrs in rail.nodes(data=True) for line in attrs['lines']})
line_rows = []
for line in line_names:
    member_ids = [n for n, attrs in rail.nodes(data=True) if line in attrs['lines']]
    member = stations[stations.station_id.isin(member_ids)]
    line_rows.append({'line': line, 'mapped_stations': len(member),
                      'mean_selected_trip_visits_per_station': member.trips_served.mean(),
                      'mean_window_departures_per_station': member.am_peak_stops.mean()})
line_summary = pd.DataFrame(line_rows)
display(line_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(line_summary.line, line_summary.mapped_stations, color='#2563eb')
axes[0].set_xlabel('Mapped stations (shared stations appear on each line)')
edges = pd.DataFrame([attrs for _, _, attrs in rail.edges(data=True)])
axes[1].hist(edges.trips / 120, bins=12, color='#08916b', edgecolor='white')
axes[1].set_xlabel('Scheduled traversals / min (both directions)')
axes[1].set_ylabel('Rail segments')
fig.tight_layout()
display(fig)
plt.close(fig)

,line,mapped_stations,mean_selected_trip_visits_per_station,mean_window_departures_per_station
0,Airport Line,20,62.90,49.50
1,Armadale Line,20,50.50,38.25
2,Ellenbrook Line,13,67.85,54.46
3,Fremantle Line,17,44.82,34.82
4,Mandurah Line,13,60.08,43.46
5,Midland Line,15,62.00,49.87
6,Thornlie-Cockburn Line,15,61.67,47.07
7,Yanchep Line,16,58.88,41.88


<Figure size 1650x600 with 2 Axes>

## Find a station for the next experiment

Use the line filter below to inspect the station table. Shared-station counts
are not additive; summing the line rows would count interchanges more than once.
The per-station frequency summary is also not the number of distinct line trips.

In [5]:
def show_line(line):
    ids = [n for n, attrs in rail.nodes(data=True)
           if line == 'All lines' or line in attrs['lines']]
    display(stations.loc[stations.station_id.isin(ids), columns]
            .sort_values(['degree', 'am_peak_stops'], ascending=False).reset_index(drop=True))

line_control = widgets.Dropdown(options=['All lines', *line_names], description='Line:')
line_output = widgets.interactive_output(show_line, {'line': line_control})
display(line_control, line_output)
print(f'Notebook computation: {time.perf_counter() - started:.2f} seconds')

Dropdown(description='Line:', options=('All lines', 'Airport Line', 'Armadale Line', 'Ellenbrook Line', 'Frema…

Output()

Notebook computation: 0.83 seconds


## What this baseline allows us to test

A tree has no alternative rail path after a corridor is cut. Node removal can
fragment it immediately, and higher capacity cannot restore the missing path.
Notebook 02 measures that structural damage. Notebook 03 tests whether a load
redistribution rule can spread failures beyond the initial closure. Notebook 04
then asks whether buses restore terminal service and how they change rail loads.

**Demo handoff:** identify a central junction on the map, then open Notebook 02.
Reproduce this notebook with `make notebooks` or `jupyter nbconvert --to notebook
--execute --inplace notebooks/01_network.ipynb`. A clean kernel needs only the
declared Python dependencies and committed files, without the raw GTFS archive.